# 8. Bahdanau Attention (Additive Attention)

In the approach by [Cho et al., 2014](https://emnlp2014.org/papers/pdf/EMNLP2014179.pdf) and [Sutskever et al., 2014](https://proceedings.neurips.cc/paper_files/paper/2014/file/5a18e133cbf9f257297f410bb7eca942-Paper.pdf), the authors employed an RNN encoder-decoder framework for neural machine translation, specifically by encoding a variable-length source sentence into a fixed-length vector. The latter would then be decoded into a variable-length target sentence.

However, [Bahdanau et al. (2014)](https://arxiv.org/pdf/1409.0473) argued that representing an entire sentence with one fixed-size vector loses information as sentences become longer. They introduced an attention mechanism that lets the decoder look at different parts of the input sentence while generating each output word, improving translation quality.

The statement is highlighted in the paper screenshot below: 

![images](./images/Problem%20with%20fixed-size%20context%20vector.png)

To better understand the motivation behind the proposed attention mechanism, we first examine the abstract of the seminal work by [Bahdanau et al. (2014)](https://arxiv.org/pdf/1409.0473), which outlines the limitations of the conventional encoder–decoder architecture and the rationale for introducing attention.

![abstract](./images/Abstract.png)

The following points are particularly significant in the Abstract:

1. Recent encoder–decoder models represent the entire source sentence using a single fixed-length vector.
2. This fixed-length vector becomes a `bottleneck`, especially for long sentences, because it cannot capture all the information in the source sentence.
3. To solve this problem, an `additive attention` mechanism was introduced that allows the decoder to automatically focus on the most relevant parts of the source sentence when predicting each target word.

**Requirements**

In [39]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

Download from the following link: 
- [https://www.manythings.org/anki/](https://download.pytorch.org/tutorial/data.zip)

Then, extract the data: `data/eng-fra.txt`. The file is a tab separated list of translation pairs. 

<p align="center">
  <img src="./images/DataPreparation.drawio.png" alt="Data Preparation">
</p>

- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [40]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input the the `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [41]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [42]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [43]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Now, let's bundle everything up as per the diagram above: 

In [44]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [45]:
PATH = r'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['il apprend le chinois', 'he s learning chinese']


15

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

## Proposed model

First, let's try to understand the proposed model mathematicallty. Later in the article we try to simplyfy the explanation using visuals, if you have already understood the mathematics, please feel free to skip the part. 

Here, the figure for the proposed model from the paper: 

![proposed-model](./images/Proposed-model-additive-attention.png)

### Encoder

![encoder](./images/Encoder-additive-attention.png)

The role of the encoder is to generate an annotation, $h_𝑖$ (concatenation of forward hidden states, and backward hidden states), for every word, $x_i$, in an input sentence of length $T$ words.

$$\mathbf{h}_i = \left[ \overrightarrow{\mathbf{h}_i^T} \; ; \; \overleftarrow{\mathbf{h}_i^T} \right]^T$$

In [46]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.


### Decoder

The decoder is responsible for generating the target sequence by selectively attending to the most relevant information encoded in the source sentence. To achieve this, it employs an attention mechanism, which enables the model to dynamically assign varying levels of importance to different parts of the input sequence during each step of the decoding process. This mechanism enhances the model's ability to capture contextual dependencies and improves the accuracy of the generated output.

Here is how the calculations in decoder occurs: 

Firstly, the goal of the decoder is to decode/predict the next word. 

To predict the next word, the following values are required: 
1. hidden states of the encoder $(h)$
2. Previous predicted word/ouput index ($y$) ($<BOS>$ index for the first time instance.)
2. hidden state of the decoder RNN of the last time step $(S_{t-1})$ 

At the **first time instance (t=1)**:

$$
s_1 = RNN(y_0, h_{T_x})
$$

where:
- $y_0$ is the index of the `<BOS>` (Beginning of Sequence) token, and
- $h_{T_x}$ is the hidden state of the last word in the source sequence.

The context vector for the first decoding step is computed using the attention mechanism:

$$
c_1 = Attention(s_1, h)
$$

or equivalently,

$$
c_1 = \sum_{j=1}^{T_x} \alpha_{1j} h_j
$$

where the attention weights are calculated as:

$$
\alpha_{1j} = \frac{\exp(e_{1j})}{\sum_{k=1}^{T_x}\exp(e_{1k})}
$$

The attention weights at the first decoding time step can be written as:

$$
\alpha_{1,i} = \text{softmax}(e_{1,i})
$$

where the attention score is defined as:

$$
e_{1,j}=a(s_1,h_j)
$$

or,

$$
e_{1,j} = v^T \tanh(W_1 h_j + W_2 s_1)
$$

The attention weights $\alpha_{1,i}$ determine the importance of each encoder hidden state $h_i$ when generating the first target word. 

Using the `decoder state` $s_1$ and the `context vector` $c_1$, 
- the decoder generates the `first target word` $y_1$.


At the **second time instance (t=2)**:

The decoder takes the previously generated word $y_1$ and the previous hidden state $s_1$ to compute the next hidden state:

$$
s_2 = RNN(y_1, s_1)
$$

where:
- $y_1$ is the target word generated at the first decoding step, and
- $s_1$ is the previous decoder hidden state.

Using the updated decoder state, the attention mechanism recalculates the attention weights over all encoder hidden states:

$$
c_2 = Attention(s_2,h)
$$

or,

$$
c_2 = \sum_{j=1}^{T_x} \alpha_{2j} h_j
$$

where the attention weights are:

$$
\alpha_{2j} = \frac{\exp(e_{2j})}{\sum_{k=1}^{T_x}\exp(e_{2k})}
$$

and the attention score is:

$$
e_{2j}=a(s_2,h_j)
$$

The decoder then generates the second target word $y_2$ using the updated hidden state $s_2$ and the context vector $c_2$.

This process continues for subsequent decoding steps, where each decoder state attends to the most relevant parts of the source sequence to generate the next target word.

----
![decoder](./images/Bahdanau-attention-decoder.png)

Now, the context vector and the encoder hidden step for this time step is concatenated, and passed through a feed-forward network.

![concatenate](./images/Concatenate.png)

- The softmax function produces a probability distribution over the target vocabulary. 
- The word corresponding to the highest probability is selected as the predicted word for the current decoding time step.

In [47]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights


# 8. Luong Attention (Multiplicative Attention)

In the approach by [Bahdanau et al. (2014)](https://arxiv.org/pdf/1409.0473), the authors introduced an additional feed-forward network to compute alignment scores between decoder and encoder states which we now refer to as `Additive Attention`. This gave the encoder-decoder architecture the ability to dynamically focus on different parts of the input sequence while generating each output token, instead of relying on a single fixed-length context vector produced by the encoder. It also allowed the model to learn soft alignments between source and target words, improving translation quality, especially for longer sequences where information was often lost in traditional sequence-to-sequence models.

To summarize, the two main contributions introduced by **Bahdanau et al. (2014)** that remain highly influential today are:

- **Replacing the single fixed-length context vector with access to all encoder hidden states.**
    - Instead of forcing the encoder to compress the entire input sequence into one vector, the decoder can access the complete set of encoder representations and retrieve relevant information when generating each output token.
- **Introducing additive attention as a mechanism for learning soft alignments.**
    - The model learns which parts of the input sequence are most relevant for each decoding step by computing attention scores between decoder and encoder states, allowing it to focus on the specific words or regions needed for generating the next token.

Together, these ideas transformed the encoder-decoder architecture from a fixed information bottleneck into a dynamic retrieval system, forming the foundation for later attention mechanisms and ultimately influencing the development of Transformer architectures.

## Luong et al. (2015), "Effective Approaches to Attention-based Neural Machine Translation"


Luong et. al., explored `more direct scoring functions` (such as `dot product`, general, and concatenation-based attention) and introduced two important design choices: 
    - `Global attention`, where the decoder considers all encoder hidden states, and 
    - `Local attention`, where it focuses only on a predicted window of source positions to reduce computation.  

### Encoder

The encoder generates an annotation ($h_i$) for each source word ($x_i$) in an input sentence of length ($T$). Each annotation is formed by concatenating the forward and backward hidden states of a bidirectional RNN:

$$
h_i = [\overrightarrow{h_i}; \overleftarrow{h_i}]
$$

where (\overrightarrow{h_i}) represents the forward hidden state and (\overleftarrow{h_i}) represents the backward hidden state.

## Decoder

The decoder functions slightly differently. 

### 1. Decoder RNN
The current decoder hidden state is computed as: 
$$ s_t = RNN_{decoder}(s_{t-1}, y_{t-1}) $$

Here, 
- $s_{t-1}$ denotes the previous hidden decoder state and 
- $y_{t-1}$ the previous decoder output. 
---

### 2. Alignment Model

The alignment model $a(\cdot)$ determines how well the current decoder state matches each encoder annotation. It computes an **alignment score** $e_{t,i}$ between the decoder hidden state at time step $t$ and the encoder hidden state corresponding to the source word $x_i$:

$$
e_{t,i} = a(s_t, h_i)
$$

where:

- $e_{t,i}$ is the **alignment score** indicating how relevant the $i$-th source word is when generating the $t$-th target word.
- $s_t$ is the current decoder hidden state, representing the decoder's current context while generating a target word.
- $h_i$ is the encoder annotation for the $i$-th source word, containing information from the encoder.
- $a(\cdot)$ is a learnable scoring function (alignment model) that measures the compatibility between $s_t$ and $h_i$.

There paper two different attention to compute the `alignment score`. 
1. Global Attention
2. Local Attention

#### **2.1. Global Attention**

The **global attention model** considers all source words in the input sentence when computing the alignment scores and, consequently, when generating the context vector.

For a target word at time step $t$, the decoder compares its current hidden state $s_t$ with every encoder hidden state $h_i$ to determine how much attention should be assigned to each source word.

Luong et al. propose three different scoring functions for computing the alignment score:

$$
e_{t,i} = a(s_t, h_i)
$$

The three approaches are:

**2.1.1. Concatenation-based Attention (similar to Bahdanau's additive attention)**

$$
a(s_t, h_i) = v_a^T \tanh(W_a[s_t;h_i])
$$

This approach concatenates the decoder hidden state $s_t$ and encoder hidden state $h_i$, then applies a feed-forward neural network to compute the alignment score.

**2.1.2 Dot-Product Attention** (most influential)

$$
a(s_t, h_i) = s_t^T h_i
$$

This approach directly computes the similarity between the decoder state and encoder state using their dot product.

**2.1.3. General Attention**

$$
a(s_t, h_i) = s_t^T W_a h_i
$$

The general approach introduces a trainable weight matrix $W_a$ before computing the dot product, allowing the model to learn a transformation between the decoder and encoder representations.

Here:

- $W_a$ is a trainable weight matrix.
- $v_a$ is a trainable weight vector.
- $[s_t;h_i]$ represents the concatenation of the decoder state and encoder annotation.

**Global Attention Intuition**

The multiplicative attention approaches (dot-product and general attention) measure the similarity between the decoder state $s_t$ and encoder state $h_i$.

The dot-product formulation:

$$
s_t^T h_i
$$

can be interpreted as asking:

> "How similar are the decoder's current representation and the representation of this source word?"

A larger dot-product value indicates that the source word represented by $h_i$ is more relevant for generating the current target word.

Compared with Bahdanau's additive attention, Luong's multiplicative attention is computationally simpler because it replaces a feed-forward network with matrix operations.

#### **2.2. Local Attention**

The **global attention model** considers all source words when computing the alignment scores and context vector. However, this can become computationally expensive for long input sequences because the decoder must compare its hidden state with every encoder annotation at each decoding step.

To address this limitation, Luong et al. introduce the **local attention model**, which computes the context vector using only a subset of encoder annotations within a fixed-size window around a predicted alignment position $p_t$.

The attention window is defined as:

$$
[p_t-D,\ p_t+D]
$$

where:

- $p_t$ is the predicted aligned source position for target time step $t$.
- $D$ is a manually selected window size.
- Only encoder annotations $h_i$ inside this window contribute to the context vector.

The local attention context vector is computed as a weighted average over the selected annotations:

$$
c_t = \sum_{i=p_t-D}^{p_t+D}\alpha_{t,i}h_i
$$

where $\alpha_{t,i}$ represents the attention weight for the source position $i$.

Luong et al. propose two methods for determining the alignment position $p_t$:

---

### 1. Monotonic Alignment

This approach assumes that source and target words are approximately aligned in the same order.

Therefore:

$$
p_t = t
$$

This works well for language pairs where word order is relatively similar.

---

### 2. Predictive Alignment

Instead of assuming a fixed alignment, the model learns to predict the source position that should be attended to.

The predicted position is computed as:

$$
p_t = S \cdot \text{sigmoid}(v_p^T \tanh(W_p s_t))
$$

where:

- $S$ is the length of the source sentence.
- $s_t$ is the current decoder hidden state.
- $W_p$ and $v_p$ are trainable parameters used to predict the aligned position.

The sigmoid function ensures that the predicted position is within the valid source sentence range:

$$
0 \leq p_t \leq S
$$

---

### 3. Attention Weights
The alignment scores are converted into attention weights using a softmax function:

$$
\alpha_{t,i} =
\frac{\exp(e_{t,i})}
{\sum_{j=1}^{T}\exp(e_{t,j})}
$$

where $\alpha_{t,i}$ represents how much attention the decoder should assign to the $i$-th source word when generating the $t$-th target word.

---

### 4. Context Vector

The context vector is then computed as a weighted sum of all encoder annotations:

$$
c_t = \sum_{i=1}^{T}\alpha_{t,i}h_i
$$

---

### 5. Attentional Hidden State 

After computing the context vector $c_t$ using the attention weights, the model combines the context vector with the current decoder hidden state $s_t$ to create an **attentional hidden state** $\tilde{s}_t$.

The attentional hidden state is computed as:

$$
\tilde{s}_t = \tanh(W_c[c_t; s_t])
$$

where:

- $\tilde{s}_t$ is the attentional hidden state used for generating the output.
- $c_t$ is the context vector obtained from the weighted sum of encoder annotations.
- $s_t$ is the current decoder hidden state.
- $[c_t; s_t]$ denotes the concatenation of the context vector and decoder hidden state.
- $W_c$ is a learnable weight matrix that transforms the concatenated representation.

The attentional hidden state combines information from both:
- the **source sentence** through the context vector $c_t$, and
- the **previously generated target sequence** through the decoder state $s_t$.

---

### 6. Output Generation

The decoder produces a final output by feeding it a weighted attentional hidden state:

$$
y_t = \text{softmax}(W_y\tilde{s}_t)
$$

where:

- $y_t$ is the predicted probability distribution for the target word at time step $t$.
- $W_y$ is the output projection matrix.
- $\tilde{s}_t$ is the attentional hidden state.

The decoding process is repeated autoregressively:

1. Update the decoder hidden state.
2. Compute alignment scores between the decoder state and encoder annotations.
3. Convert alignment scores into attention weights.
4. Compute the context vector.
5. Generate the attentional hidden state.
6. Predict the next target word.

These steps are repeated until the end-of-sequence token is generated.

---

![comparision with Bahdanau Attention](https://machinelearningmastery.com/wp-content/uploads/2021/10/luong_1-768x320.png)
Source: [Advanced Deep Learning with Python](https://www.amazon.com/Advanced-Deep-Learning-Python-next-generation/dp/178995617X)

![Luong_output](./images/Luong_ouptput.png)

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

`Question`: Do you think we need to change the decoder code too? 

## Summary

- The goal of Luong attention (from the paper ["Effective Approaches to Attention-based Neural Machine Translation"](https://arxiv.org/pdf/1508.04025) by Thang Luong, Hieu Pham, and Christopher Manning) was to make attention mechanisms simpler, faster, and more effective for neural machine translation after Bahdanau attention showed that dynamic alignment was valuable. 
- Bahdanau attention introduced an additional feed-forward network to compute alignment scores between decoder and encoder states, 
- Luong et. al., explored `more direct scoring functions` (such as `dot product`, general, and concatenation-based attention) and introduced two important design choices: 
    - `Global attention`, where the decoder considers all encoder hidden states, and 
    - `Local attention`, where it focuses only on a predicted window of source positions to reduce computation. 
- The main improvement was that Luong attention provided a more efficient attention mechanism with competitive or better translation performance, especially when combined with stronger recurrent architectures like multi-layer LSTMs. 
- Luong showed that attention could be computed through direct vector similarity, making it simpler and computationally efficient. (`multiplicative global attention using dot product`)
- This idea became highly influential because it established the foundation for modern attention mechanisms: representing attention as `query-key` matching followed by a weighted sum of values. 
- The Transformer architecture later generalized this idea into scaled dot-product attention, where queries, keys, and values are learned projections of the input representations. 
- Therefore, while dynamic context retrieval (`local attention`) was the conceptual breakthrough of attention in general, Luong's multiplicative/dot-product attention (*global attention*) is the specific mechanism that most directly influenced the attention operation used in today's large language models.

### References:

- [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/pdf/1508.04025)
- [Machine Learning Mastery](https://machinelearningmastery.com/the-luong-attention-mechanism/)
- [Jay Allamar's Blog](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/)

## Training


## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [49]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [50]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [51]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [52]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [53]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [54]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [55]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating


In [56]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 5s (- 3m 42s) (5 2%) 1.8357
0m 11s (- 3m 31s) (10 5%) 1.1070
0m 15s (- 3m 17s) (15 7%) 0.7941
0m 20s (- 3m 8s) (20 10%) 0.5358
0m 25s (- 2m 58s) (25 12%) 0.3531
0m 30s (- 2m 50s) (30 15%) 0.2369
0m 34s (- 2m 44s) (35 17%) 0.1608
0m 39s (- 2m 38s) (40 20%) 0.1204
0m 44s (- 2m 33s) (45 22%) 0.0964
0m 49s (- 2m 27s) (50 25%) 0.0794
0m 54s (- 2m 22s) (55 27%) 0.0698
0m 59s (- 2m 17s) (60 30%) 0.0646
1m 3s (- 2m 12s) (65 32%) 0.0603
1m 8s (- 2m 7s) (70 35%) 0.0559
1m 13s (- 2m 2s) (75 37%) 0.0545
1m 18s (- 1m 57s) (80 40%) 0.0528
1m 23s (- 1m 52s) (85 42%) 0.0493
1m 28s (- 1m 47s) (90 45%) 0.0502
1m 32s (- 1m 42s) (95 47%) 0.0481
1m 37s (- 1m 37s) (100 50%) 0.0476
1m 42s (- 1m 32s) (105 52%) 0.0464
1m 47s (- 1m 28s) (110 55%) 0.0449
1m 52s (- 1m 23s) (115 57%) 0.0446
1m 57s (- 1m 18s) (120 60%) 0.0458
2m 2s (- 1m 13s) (125 62%) 0.0432
2m 6s (- 1m 8s) (130 65%) 0.04

In [57]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis suffisamment grand
= i m old enough
< i m old enough <EOS>

> tu caches quelque chose
= you re hiding something
< you re hiding something <EOS>

> nous avons honte
= we re ashamed
< we re ashamed <EOS>

> vous etes fort effrontes
= you re very forward
< you re very forward <EOS>

> je suis honnete
= i m honest
< i m honest <EOS>

> vous etes branche
= you re fashionable
< you re my right <EOS>

> tu es intrepide
= you re fearless
< you re fearless <EOS>

> je songe a demissionner
= i m considering resigning
< i m considering resigning <EOS>

> je suis sincerement desolee
= i m truly sorry
< i m truly sorry <EOS>

> vous etes pardonnees
= you re forgiven
< you re forgiven <EOS>



`Note`: Could train this with varying degree of success. 

- The goal of Bahdanau attention (from the paper ["Neural Machine Translation by Jointly Learning to Align and Translate"](https://arxiv.org/pdf/1409.0473) by Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio) was to solve the main weakness of early encoder-decoder translation models: the fixed-length bottleneck. Before attention, the encoder compressed an entire source sentence into one final hidden vector, forcing the decoder to extract all information from that single representation; this became especially problematic for long sentences because important details were lost. 
- Bahdanau attention introduced a mechanism where the decoder could dynamically look back at all encoder hidden states and learn which source words are relevant at each generation step, effectively creating a soft alignment between input and output words (e.g., when generating "house" in English, the model can attend strongly to "maison" in French). This was `Additive Attention`. 
- The improvement was that translation quality increased significantly, especially for longer sentences, because the model no longer needed to memorize the entire sentence in one vector—it could retrieve information when needed. 
- In the current context, the core idea of Bahdanau attention remains extremely important because it introduced the concept of selective information retrieval, which became the foundation for modern Transformer architectures: Transformers replaced recurrent attention with self-attention, allowing every token to directly interact with every other token in parallel, but the fundamental idea remains the same
    - do not compress everything into one hidden state; instead, learn what information is important for each prediction step.

---
### References:

- [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/pdf/1409.0473)
- [Machine Learning Mastery](https://machinelearningmastery.com/the-bahdanau-attention-mechanism/)
- [Dive into Deep Learning](https://d2l.ai/chapter_attention-mechanisms-and-transformers/bahdanau-attention.html)
- [Pytorch Documentation](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html)

# Lab Report: Additive and Multiplicative Attention Mechanisms

## 1. Objective
To implement and compare Bahdanau (Additive) Attention and Luong (Multiplicative) Attention mechanisms to understand how dynamic context vectors solve the information bottleneck in Sequence-to-Sequence (Seq2Seq) models.

## 2. Methodology
The dataset (French-to-English translation) was preprocessed and tokenized. An Encoder-Decoder RNN architecture was utilized. Instead of using a single fixed context vector from the Encoder, we implemented:
- **Additive Attention (Bahdanau):** Uses a feed-forward network to calculate alignment scores between decoder and encoder states.
- **Multiplicative Attention (Luong):** Computes direct vector similarity (dot product) to find alignment scores, which is computationally simpler.

The models were put into evaluation mode using `.eval()` (disabling layers like Dropout) to perform deterministic inference and evaluate translation accuracy.

## 3. Results
During the evaluation phase using `evaluateRandomly()`, the model successfully translates sentences from French to English. The output logs provide a comparison:
- `>` represents the input French sentence.
- `=` represents the target English translation (ground truth).
- `<` represents the model's predicted English translation.

Even without understanding French, comparing the target (`=`) and predicted (`<`) sentences confirms the model's accuracy. The model quickly learns to map the representations and outputs near-perfect translations on the small, filtered dataset.

## 4. Discussion
Both attention mechanisms effectively solve the fixed-length bottleneck of traditional Seq2Seq models. By generating a context vector that dynamically focuses on relevant source words at each generation step, the model preserves details for long sentences. Luong's multiplicative approach simplifies the computation over Bahdanau's additive approach while maintaining strong translation performance, establishing the "query-key" dot-product foundation used in modern architectures like the Transformer.

## 5. Conclusion
In this lab, the attention mechanism was successfully integrated into an RNN Encoder-Decoder model. The dynamic alignment of source and target tokens significantly improves translation quality, highlighting the critical shift from static representation to selective information retrieval in deep learning NLP models.